# Partie 1 — Mise en place du cluster et positionnement

**Objectif :** vérifier que l'environnement Docker Compose fonctionne et que la `SparkSession`
est bien connectée au Master `spark://spark-master:7077`.

| Service | URL (depuis Windows) |
|---|---|
| Spark Master UI (2 Workers attendus) | http://localhost:8090 |
| Spark UI de l'application (Driver) | http://localhost:4040 |
| Console MinIO (bucket `lake`) — `minioadmin` / `minioadmin` | http://localhost:9001 |
| Jupyter | http://localhost:8888 |

## 1. SparkSession connectée au Master

In [1]:
import time
from spark_session import get_spark, LAKE

spark = get_spark("FastFood-Partie1")
sc = spark.sparkContext

# Les Executors s'enregistrent quelques secondes après la création de la session :
# on attend les 2 Executors (+ le Driver) avant de lire defaultParallelism.
for _ in range(60):
    if sc._jsc.sc().getExecutorMemoryStatus().size() >= 3:
        break
    time.sleep(1)

print("Version de Spark   :", spark.version)
print("Master             :", sc.master)
print("appName            :", sc.appName)
print("Coeurs disponibles : sc.defaultParallelism =", sc.defaultParallelism)
print("Application ID     :", sc.applicationId)

Version de Spark   : 3.5.3
Master             : spark://spark-master:7077
appName            : FastFood-Partie1
Coeurs disponibles : sc.defaultParallelism = 4
Application ID     : app-20261121172622-0000


## 2. Driver et Executors de l'application

In [2]:
infos = sc._jsc.sc().statusTracker().getExecutorInfos()
print(f"{'Processus':<12} {'Hôte':<16} Port")
for e in infos:
    role = "Driver" if e.host() == "jupyter" else "Executor"
    print(f"{role:<12} {e.host():<16} {e.port()}")

Processus    Hôte             Port
Driver       jupyter          41939
Executor     172.19.0.8       40447
Executor     172.19.0.7       44857


## 3. Un job distribué : quelle machine traite quelle partition ?

Chaque partition est une *tâche* exécutée par un Executor sur un Worker. Le Driver (Jupyter)
ne fait que planifier et récupérer le résultat.

In [3]:
import socket

rdd = sc.parallelize(range(1, 1001), 8)
placement = rdd.mapPartitionsWithIndex(
    lambda idx, it: [(idx, socket.gethostname(), sum(it))]
).collect()

for idx, host, s in placement:
    print(f"partition {idx} -> exécutée sur {host:<15} somme partielle = {s}")
print("Somme totale :", sum(s for _, _, s in placement))

partition 0 -> exécutée sur spark-worker-2  somme partielle = 7875
partition 1 -> exécutée sur spark-worker-2  somme partielle = 23500
partition 2 -> exécutée sur spark-worker-1  somme partielle = 39125
partition 3 -> exécutée sur spark-worker-1  somme partielle = 54750
partition 4 -> exécutée sur spark-worker-2  somme partielle = 70375
partition 5 -> exécutée sur spark-worker-2  somme partielle = 86000
partition 6 -> exécutée sur spark-worker-1  somme partielle = 101625
partition 7 -> exécutée sur spark-worker-1  somme partielle = 117250
Somme totale : 500500


## 4. Data Lake MinIO : le bucket `lake` est accessible depuis Spark (S3A)

In [4]:
jvm = spark._jvm
root = jvm.org.apache.hadoop.fs.Path(LAKE + "/")
fs = root.getFileSystem(sc._jsc.hadoopConfiguration())
print("Bucket", LAKE, "existe :", fs.exists(root))
for st in fs.listStatus(root):
    print("  ", st.getPath().toString())

Bucket s3a://lake existe : True


   s3a://lake/bronze


## 5. Kafka : le topic `orders` existe avec 3 partitions

In [5]:
%pip install -q "kafka-python>=2.1"

Note: you may need to restart the kernel to use updated packages.


In [6]:
from kafka import KafkaConsumer
consumer = KafkaConsumer(bootstrap_servers="kafka:9092")
print("Topics :", sorted(t for t in consumer.topics() if not t.startswith("__")))
print("Partitions du topic orders :", sorted(consumer.partitions_for_topic("orders")))
consumer.close()

Topics : ['orders']
Partitions du topic orders : [0, 1, 2]


## 6. Captures à faire pour le rapport

1. http://localhost:8090 → **2 Workers ALIVE** et l'application `FastFood-Partie1` en cours.
2. http://localhost:9001 → bucket **`lake`**.
3. La sortie de la cellule 1 (version, master, appName, defaultParallelism).

Ensuite, libérer les ressources du cluster :

In [7]:
spark.stop()